# Bulk RNA-seq Pipeline — Colab

Run the full Snakemake pipeline (DESeq2 · GSEA · ORA · TFEA · PROGENy · cMap → HTML report) in the browser. No local Docker or WSL required.

**What you need:**
- `salmon.merged.gene_counts_length_scaled.tsv` from a prior nf-core/rnaseq run
- `multiqc_data/` directory from the same run, **zipped** (Colab uploads accept files, not folders)

**What you'll get:** `results.zip` containing `report/report.html` and all intermediate tables.

**Time budget:** ~10 minutes on first run (env setup dominates). Rerunning within the same session: 1–2 min.

**Run order:** top to bottom. Each section has a short description; edit only the cells marked `### ← EDIT ###`.

## 1. Install conda + Snakemake

One-time setup for this session. Miniforge ships mamba (fast conda).

In [ ]:
%%time
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
!bash /tmp/miniforge.sh -b -p /content/miniforge
!rm /tmp/miniforge.sh

# Persist miniforge on PATH so Snakemake subprocesses find `conda`.
import os
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

!mamba install -y -n base -c conda-forge -c bioconda snakemake pandas pyyaml 2>&1 | tail -5
!snakemake --version

## 2. Get the pipeline code

In [ ]:
!rm -rf /content/bulk-rnaseq
!git clone --depth=1 https://github.com/artblakey19/rnaseq-report-MMB.git /content/bulk-rnaseq
%cd /content/bulk-rnaseq

## 3. Upload your data

Two uploads:

1. **Counts TSV** — `salmon.merged.gene_counts_length_scaled.tsv` (single file)
2. **MultiQC data** — `multiqc_data/` zipped into a single `.zip` (Colab cannot accept folders directly)

   On your machine: `zip -r multiqc_data.zip multiqc_data/` before uploading.

**Want to try it first without your own data?** Skip section 3 and run the "Use test fixture" cell at the bottom of section 4 instead.

In [ ]:
# Upload counts TSV
from google.colab import files
uploaded = files.upload()
counts_file = next(iter(uploaded))
!mv "{counts_file}" /content/bulk-rnaseq/counts.tsv
!ls -la counts.tsv
print('Header:')
!head -1 counts.tsv | tr '\t' '\n'

In [ ]:
# Upload multiqc_data.zip and extract
from google.colab import files
uploaded = files.upload()
zip_file = next(iter(uploaded))
!rm -rf multiqc_data
!unzip -q "{zip_file}" -d /tmp/mqc_unpack
# Handle both "multiqc_data.zip containing multiqc_data/" and "zip of file contents directly"
!if [ -d /tmp/mqc_unpack/multiqc_data ]; then mv /tmp/mqc_unpack/multiqc_data /content/bulk-rnaseq/; else mv /tmp/mqc_unpack /content/bulk-rnaseq/multiqc_data; fi
!rm -f "{zip_file}"
!ls multiqc_data | head -10

## 4. Project metadata  &nbsp; `### ← EDIT ###`

Fill in the form. These fields appear in the report header (reported only, not used in any calculation).

In [ ]:
#@markdown ### Project
project_name = "My RNA-seq study"  #@param {type:"string"}
analyst = ""  #@param {type:"string"}

#@markdown ### Experiment context (all optional)
cell_line = ""  #@param {type:"string"}
compound = ""  #@param {type:"string"}
dose = ""  #@param {type:"string"}
duration = ""  #@param {type:"string"}
notes = ""  #@param {type:"string"}

print('project metadata captured.')

## 5. Samples and contrasts  &nbsp; `### ← EDIT ###`

The next cell **detects sample names from your counts file** and prints a template you can paste into the cell below it.

Edit the `SAMPLES` dict so each sample maps to its **condition** (control/treatment/whatever grouping you want to compare). Then list the comparisons you care about in `CONTRASTS`.

In [ ]:
# Detect samples from counts header (columns 3+ are samples)
from pathlib import Path

counts_path = Path('/content/bulk-rnaseq/counts.tsv')
if not counts_path.exists():
    # Fall back to test fixture for users who skipped upload
    counts_path = Path('/content/bulk-rnaseq/tests/test_data/counts.tsv')
    print('(using test fixture — no user data uploaded)')

header = counts_path.read_text().splitlines()[0].split('\t')
detected_samples = header[2:]
print(f'Detected {len(detected_samples)} samples:\n')
print('SAMPLES = {')
for s in detected_samples:
    print(f'    {s!r}: "CHANGE_ME",')
print('}')

In [ ]:
### ← EDIT ### — paste the template from the cell above, then set each condition.
SAMPLES = {
    "Control_1":   "Control",
    "Control_2":   "Control",
    "Treatment_1": "Treatment",
    "Treatment_2": "Treatment",
}

### ← EDIT ### — list every comparison you want (numerator vs denominator).
# numerator = the condition of interest (e.g. treated)
# denominator = the reference (e.g. control)
CONTRASTS = [
    # (contrast_id, numerator, denominator, description)
    ("Treatment_vs_Control", "Treatment", "Control", "Treatment vs Control"),
]

## 6. Generate config files

Writes `config/config.yaml`, `config/samples.tsv`, `config/contrasts.tsv` from the form inputs + SAMPLES/CONTRASTS above. Uses the committed `config.template.yaml` as the base for all analysis settings (DE cutoffs, MSigDB collections, etc.).

In [ ]:
import csv
from collections import Counter
from pathlib import Path
import yaml

repo = Path('/content/bulk-rnaseq')
config_path = repo / 'config' / 'config.yaml'
samples_path = repo / 'config' / 'samples.tsv'
contrasts_path = repo / 'config' / 'contrasts.tsv'
template_path = repo / 'config' / 'config.template.yaml'

# Decide which counts/multiqc to use
user_counts = repo / 'counts.tsv'
user_mqc = repo / 'multiqc_data'
if user_counts.exists() and user_mqc.exists():
    counts_rel = 'counts.tsv'
    mqc_rel = 'multiqc_data'
    print('Using uploaded data.')
else:
    counts_rel = 'tests/test_data/counts.tsv'
    mqc_rel = 'tests/test_data/multiqc_data'
    print('Using bundled test fixture.')

cfg = yaml.safe_load(template_path.read_text())
cfg['project']['name'] = project_name
cfg['project']['analyst'] = analyst
cfg['project']['experiment'] = {
    'cell_line': cell_line, 'compound': compound, 'dose': dose,
    'duration': duration, 'notes': notes,
}
cfg['input']['counts_tsv'] = counts_rel
cfg['input']['multiqc_data_dir'] = mqc_rel
cfg['input']['samples_tsv'] = 'config/samples.tsv'
cfg['input']['contrasts_tsv'] = 'config/contrasts.tsv'

# Validate SAMPLES against counts header
header = (repo / counts_rel).read_text().splitlines()[0].split('\t')[2:]
missing = [s for s in header if s not in SAMPLES]
extra = [s for s in SAMPLES if s not in header]
if missing or extra:
    raise SystemExit(
        f'SAMPLES dict mismatch.\n'
        f'  samples in counts but not in SAMPLES: {missing}\n'
        f'  keys in SAMPLES but not in counts:   {extra}'
    )
unknown = [c for c in SAMPLES.values() if c == 'CHANGE_ME' or not c]
if unknown:
    raise SystemExit('fill in every SAMPLES value (no CHANGE_ME / empty allowed)')

# samples.tsv — auto-number replicates within each condition
replicate_of = Counter()
sample_rows = []
for sid in header:  # preserve column order
    cond = SAMPLES[sid]
    replicate_of[cond] += 1
    sample_rows.append({'sample': sid, 'condition': cond,
                         'replicate': replicate_of[cond], 'batch': 1})

# contrasts.tsv
conds = set(SAMPLES.values())
contrast_rows = []
for cid, num, den, desc in CONTRASTS:
    if num not in conds or den not in conds:
        raise SystemExit(f'contrast {cid}: {num!r} or {den!r} not in conditions {conds}')
    contrast_rows.append({'contrast_id': cid, 'factor': 'condition',
                           'numerator': num, 'denominator': den, 'description': desc})

config_path.parent.mkdir(parents=True, exist_ok=True)
with config_path.open('w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, default_flow_style=False)

def write_tsv(p, rows, cols):
    with p.open('w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=cols, delimiter='\t')
        w.writeheader(); w.writerows(rows)

write_tsv(samples_path, sample_rows, ['sample', 'condition', 'replicate', 'batch'])
write_tsv(contrasts_path, contrast_rows,
          ['contrast_id', 'factor', 'numerator', 'denominator', 'description'])

print('wrote:')
print(f'  {config_path.relative_to(repo)}')
print(f'  {samples_path.relative_to(repo)} ({len(sample_rows)} rows)')
print(f'  {contrasts_path.relative_to(repo)} ({len(contrast_rows)} rows)')

## 7. Dry-run (verifies the DAG without running anything)

In [ ]:
!snakemake \
  --snakefile workflow/Snakefile \
  --configfile config/config.yaml \
  --dry-run 2>&1 | tail -30

## 8. Run the pipeline

First run: ~10 minutes (most of it is per-rule conda env installation). Rerunning within the same Colab session: 1–2 minutes (envs are cached in `.snakemake/conda/`). `--keep-going` continues past soft failures (e.g. L2S2 network hiccup).

In [ ]:
%%time
!snakemake \
  --snakefile workflow/Snakefile \
  --configfile config/config.yaml \
  --use-conda \
  --cores 2 \
  --keep-going 2>&1 | tail -80

## 9. Download results

Bundles everything under `results/` into a single zip. The HTML report is at `report/report.html` inside it.

In [ ]:
from pathlib import Path
from google.colab import files

report = Path('results/report/report.html')
if not report.exists():
    raise SystemExit('report.html missing — check section 8 output or section 10 logs')

!zip -qr results.zip results/
print(f'report size: {report.stat().st_size/1024:.1f} KB')
files.download('results.zip')

## 10. Troubleshooting

Run this only if section 8 errored. Dumps every per-rule log so you can see which rule failed and why.

In [ ]:
!for f in $(find logs -name '*.log' 2>/dev/null); do echo "=== $f ==="; tail -30 "$f"; echo; done